# 03 · RQ2 and RQ3: advice quality (H2a, H3b)

Analysis plan §4.1 and §5. Author ratings of the P0-level replies (community, Gemma-P0, SEA-LION-P0): `rating ~ source + rater + (1 | post)`, effects in rating points.

In [ ]:
# Shared setup: every path comes from payo_eval.paths, so PAYO_* variables can point the notebook at synthetic data
import warnings

import matplotlib.pyplot as plt
import pandas as pd

from payo_eval import analyses as A
from payo_eval import load
from payo_eval.paths import get_paths, load_config

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

In [ ]:
# Settings from 03-evaluation/config.toml; outputs go to results/03-advice-quality/ and figures/
PATHS = get_paths()
CFG = load_config(PATHS)
SEED = CFG["seed"]
B = CFG["analysis"]["bootstrap_b"]
RESULTS = PATHS.results / "03-advice-quality"
TESTS = PATHS.results / "tests"
FIGURES = PATHS.figures
for folder in (RESULTS, TESTS, FIGURES):
    folder.mkdir(parents=True, exist_ok=True)


def save(table: pd.DataFrame, stem: str) -> pd.DataFrame:
    """Write a result table as CSV and return it for display."""
    table.to_csv(RESULTS / f"{stem}.csv", index=False)
    return table

In [ ]:
# Every rating joined with the blinding key and gold labels
ratings = load.ratings(PATHS)
best = load.bestfit(PATHS)
print(len(ratings), "ratings of", ratings["item_id"].nunique(), "items on", ratings["post_id"].nunique(), "posts")

## H2a: community vs. LLM

Cultural fit and empathy: directional (community higher). Helpfulness: non-inferiority of each LLM with margin Δ. The table also has the length-adjusted refit (v1.1).

In [ ]:
tests, contrasts = A.advice_vs_community(ratings, CFG["analysis"]["margin_points"])
tests.to_csv(TESTS / "F2_h2a.csv", index=False)
tests

In [ ]:
save(contrasts, "h2a_contrasts")

## Confounds: reply length and upvotes

In [ ]:
save(A.length_by_source(ratings), "reply_length_by_source")

In [ ]:
save(A.upvote_confound(ratings), "upvote_confound")

## Best overall fit (primary choices)

In [ ]:
save(A.best_fit_rates(best), "best_fit")

In [ ]:
# Mean rating per source and dimension, for the figure
means = A.p0_level(ratings).groupby("source")[list(A.DIMENSIONS)].mean()
ax = means.T.plot.bar(figsize=(8, 4), ylim=(1, 5), rot=0)
ax.set_ylabel("mean rating")
ax.figure.tight_layout()
ax.figure.savefig(FIGURES / "03_mean_ratings.png", dpi=150)
means

## H3b: SEA-LION vs. Gemma

Cultural fit and register (directional), and the safe-escalation TOST (descriptive if fewer than 20 posts need escalation).

In [ ]:
tests_h3b, escalation = A.model_comparison(ratings, CFG["analysis"]["margin_escalation"],
                                          CFG["analysis"]["min_escalation_posts"])
tests_h3b.to_csv(TESTS / "F3_h3b.csv", index=False)
tests_h3b

In [ ]:
save(escalation, "escalation")

In [ ]:
save(A.harm_rates(ratings), "harm_flags")

## Findings

_Fill in after the real run: the main estimates with their CIs, and anything flagged above._